In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial import cKDTree

from generate_cubed_sphere import EARTH_RADIUS_M
from set_up_secs import SECSFit

DATA_PATH = (
    Path.cwd().parent
    / "data"
    / "all_stations"
    / "SuperMAG_60s_20231201_rev-0006.1783148935.netcdf"
)

fit = SECSFit(DATA_PATH)
fit._make_grid()

grid = fit.pole_grid
eval_grid = fit.eval_grid

print(grid)
print("ionospheric height (m):", grid.iono_height)
print("radius (m):", grid.radius)
print("shape:", grid.shape)
print("points inside box:", grid.inside_box.sum())
print("central cell size (m):", grid.radius * grid.dxi)
print()
print(eval_grid)
print("shape:", eval_grid.shape)
print("points inside box:", eval_grid.inside_box.sum())

In [ ]:
mask1 = grid.inside_box
mask2 = eval_grid.inside_box

lon1 = grid.lon[mask1] % 360
lat1 = grid.lat[mask1]
lon2 = eval_grid.lon[mask2] % 360
lat2 = eval_grid.lat[mask2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(lon1, lat1, s=6, color="red", label="pole grid")
axes[0].scatter(lon2, lat2, s=6, color="green", label="eval grid")
axes[0].set_xlabel("Longitude (deg)")
axes[0].set_ylabel("Latitude (deg)")
axes[0].set_xlim(0, 360)
axes[0].set_ylim(min(lat1.min(), lat2.min()) - 2, max(lat1.max(), lat2.max()) + 2)
axes[0].grid(True)
axes[0].legend()

axes[1].scatter(grid.xi[mask1], grid.eta[mask1], s=6, color="red", label="pole grid")
axes[1].scatter(eval_grid.xi[mask2], eval_grid.eta[mask2], s=6, color="green", label="eval grid")
axes[1].set_xlabel("xi (rad)")
axes[1].set_ylabel("eta (rad)")
axes[1].set_aspect("equal")
axes[1].grid(True)
axes[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
def grid_xyz(grid):
    mask = grid.inside_box
    lon = np.deg2rad(grid.lon[mask] % 360)
    lat = np.deg2rad(grid.lat[mask])
    r = grid.radius
    return (
        r * np.cos(lat) * np.cos(lon),
        r * np.cos(lat) * np.sin(lon),
        r * np.sin(lat),
    )

u = np.linspace(0, 2 * np.pi, 60)
v = np.linspace(0, np.pi, 30)
sphere_x = EARTH_RADIUS_M * np.outer(np.cos(u), np.sin(v))
sphere_y = EARTH_RADIUS_M * np.outer(np.sin(u), np.sin(v))
sphere_z = EARTH_RADIUS_M * np.outer(np.ones_like(u), np.cos(v))

def add_sphere(ax):
    ax.plot_surface(sphere_x, sphere_y, sphere_z, color="lightblue", alpha=0.2, linewidth=0)

def add_points(ax, points, color, label):
    ax.scatter(*points, s=8, color=color, label=label)

def setup_3d_axes(ax):
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_zlabel("z (m)")
    ax.set_box_aspect([1, 1, 1])
    ax.view_init(elev=30, azim=-60)

In [ ]:
pole_points = grid_xyz(grid)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
add_sphere(ax)
add_points(ax, pole_points, "red", "pole grid")
setup_3d_axes(ax)
ax.legend()
plt.show()
print("red points:", pole_points[0].size)

In [ ]:
eval_points = grid_xyz(eval_grid)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
add_sphere(ax)
add_points(ax, eval_points, "green", "eval grid")
setup_3d_axes(ax)
ax.legend()
plt.show()
print("green points:", eval_points[0].size)

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection="3d")
add_sphere(ax)
add_points(ax, pole_points, "red", "pole grid")
add_points(ax, eval_points, "green", "eval grid")
setup_3d_axes(ax)
ax.legend()
plt.show()
print("red points:", pole_points[0].size)
print("green points:", eval_points[0].size)

In [ ]:
if not np.isclose(grid.radius, eval_grid.radius):
    print("grids are at different heights, so they cannot intersect")
else:
    lon1 = np.deg2rad(grid.lon[grid.inside_box])
    lat1 = np.deg2rad(grid.lat[grid.inside_box])
    lon2 = np.deg2rad(eval_grid.lon[eval_grid.inside_box])
    lat2 = np.deg2rad(eval_grid.lat[eval_grid.inside_box])

    r = grid.radius
    points1 = np.column_stack((
        r * np.cos(lat1) * np.cos(lon1),
        r * np.cos(lat1) * np.sin(lon1),
        r * np.sin(lat1),
    ))
    points2 = np.column_stack((
        r * np.cos(lat2) * np.cos(lon2),
        r * np.cos(lat2) * np.sin(lon2),
        r * np.sin(lat2),
    ))

    tree = cKDTree(points1)
    distances, indices = tree.query(points2)
    tolerance = 1.0
    common = distances < tolerance

    print("grid points:", len(points1), "and", len(points2))
    print("intersecting points:", common.sum())
    print("minimum distance (m):", distances.min())